# Set Up

In [1]:
import numpy as np
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.processamento import decompor_estatisticas
from pcj.metricas_avaliacao import avaliacao_completa, metricas_juntando

In [2]:
# Base não tratada: as médias mensais da base limpa, com os outliers (como em aplicacoes/outliers)
df, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")
df_med = decompor_estatisticas(df, estatistica="med")

# Base tratada: aqui, só para saber quais meses completos ela tem (ver "Mesma prova")
df_tratada, _, _ = dados(r"data\processed\dados_med_tratados.xlsx")

# Mesma prova
A base não tratada tem mais meses completos que a tratada: os meses que tiveram valores tratados ficaram com NaN e saem do `dropna()`. Para comparar as duas bases com a mesma prova, o sorteio fica só nos meses completos da tratada (`meses_sorteio`). Os outros meses completos da não tratada continuam na base que o imputador vê, mas não são sorteados. Com as mesmas seeds, as células sorteadas são as mesmas do notebook `imputacoes_outliers_tratados`.

In [3]:
# Meses completos (sem NaN) de cada base
completos = df_med.dropna()
completos_tratada = df_tratada.dropna()
print("Meses completos, não tratada:", len(completos))
print("Meses completos, tratada:", len(completos_tratada))

# Os índices das duas bases são diferentes (na tratada, lida do Excel, vão de 0 a 190),
# por isso os meses são casados pela data. isin marca True nas datas que estão na outra tabela
na_tratada = completos["Data"].isin(completos_tratada["Data"])
meses_sorteio = completos.index[na_tratada]
print("Meses que podem ser sorteados:", len(meses_sorteio))

# Meses completos só na não tratada (o ~ inverte True e False): ficam na base, mas não são sorteados
completos.loc[~na_tratada, "Data"].dt.strftime("%Y-%m").tolist()

Meses completos, não tratada: 126
Meses completos, tratada: 118
Meses que podem ser sorteados: 118


['2009-12',
 '2014-02',
 '2014-07',
 '2014-09',
 '2014-11',
 '2015-11',
 '2016-02',
 '2022-10']

# Experimento A
Base não tratada (médias mensais, com os outliers), com a mesma prova da tratada. O resto é igual ao notebook `imputacoes_outliers_tratados`: média e mediana com 5% de falhas e KNN com 30%, em 30 rodadas (seeds de 0 a 29).

In [4]:
# tabela: uma linha por seed × método × variável (as métricas de cada rodada)
# celulas: uma linha por célula avaliada (valor verdadeiro e imputado), usada para juntar as rodadas
tabela, celulas = avaliacao_completa(df_med, n_seeds=30, devolver_celulas=True, meses_sorteio=meses_sorteio)
tabela.head()

,variavel,rmse,mae,r2,n_celulas,metodo,pct_faltantes,seed
0,COR,69.716523,65.687603,-0.000855,5,media,0.05,0
1,TURB.,53.440060,46.449587,-0.010502,5,media,0.05,0
2,pH,0.141906,0.115207,-0.936277,5,media,0.05,0
3,ALC.,16.653125,15.920661,-1.602539,5,media,0.05,0
4,AC.,3.635749,2.775207,-0.197344,5,media,0.05,0


Conferência: número de rodadas, células fora dos meses do sorteio (precisa ser 0) e células avaliadas por variável em cada rodada (as mesmas da tratada: 5 e 35)

In [5]:
print("Rodadas:", tabela["seed"].nunique())

# A coluna indice guarda o rótulo da linha de cada célula sorteada
print("Células fora dos meses do sorteio:", (~celulas["indice"].isin(meses_sorteio)).sum())

# Células avaliadas por variável em cada rodada. O número é o mesmo em todas as rodadas,
# por isso basta a primeira linha de cada método (first)
tabela.groupby("metodo", sort=False)[["pct_faltantes", "n_celulas"]].first()

Rodadas: 30
Células fora dos meses do sorteio: 0


,pct_faltantes,n_celulas
metodo,,
media,0.05,5
mediana,0.05,5
knn,0.30,35


# Métricas juntando as rodadas
Como no notebook da tratada: RMSE, MAE e R² calculados uma vez sobre todas as células das 30 rodadas (`metricas_juntando`), e o DP do RMSE e do MAE entre as rodadas.

In [6]:
juntando = metricas_juntando(celulas)

# DP entre as rodadas: quanto o RMSE e o MAE variam de uma rodada (seed) para outra.
# O groupby separa a tabela por método e variável; o std calcula o DP das 30 rodadas de cada grupo
desvio = tabela.groupby(["metodo", "variavel"], sort=False)[["rmse", "mae"]].std()

# add_suffix renomeia as colunas (rmse -> rmse_dp, mae -> mae_dp).
# reset_index devolve "metodo" e "variavel", que o groupby transformou em índice, para colunas comuns
desvio = desvio.add_suffix("_dp").reset_index()

# merge junta as duas tabelas pelas linhas com o mesmo método e a mesma variável
resumo = juntando.merge(desvio, on=["metodo", "variavel"])
resumo = resumo[["metodo", "pct_faltantes", "variavel", "rmse", "rmse_dp", "mae", "mae_dp",
                 "r2", "n_celulas", "n_seeds"]]
resumo

,metodo,pct_faltantes,variavel,rmse,rmse_dp,mae,mae_dp,r2,n_celulas,n_seeds
0,media,0.05,COR,85.985252,30.694843,68.989642,22.668821,-0.037296,150,30
1,media,0.05,TURB.,70.764762,23.185279,57.400551,16.646832,-0.011247,150,30
2,media,0.05,pH,0.152779,0.050763,0.122854,0.043294,-0.020539,150,30
3,media,0.05,ALC.,19.482914,5.029581,16.739063,4.683499,-0.080192,150,30
4,media,0.05,AC.,5.180481,2.237031,3.812562,1.685738,-0.014401,150,30
5,media,0.05,O.C.,1.877825,0.627044,1.513642,0.563493,-0.083706,150,30
6,media,0.05,O.D.,0.822210,0.299098,0.653322,0.274945,-0.044833,150,30
7,media,0.05,Cl,14.358410,4.733643,11.532121,3.419074,-0.098428,150,30
8,media,0.05,DUR.,14.855561,4.628657,12.275372,3.974618,-0.021991,150,30
9,media,0.05,Fe,1.663910,0.608167,1.353794,0.509619,-0.038046,150,30


# Comparação: tratada × não tratada
Os resultados da tratada vêm do Excel gravado pelo notebook `imputacoes_outliers_tratados` (se algo mudar lá, rode aquele antes deste). Uma tabela por métrica: variáveis nas linhas; para cada método, o valor na tratada, na não tratada e a diferença (tratada − não tratada).
- RMSE e MAE: diferença negativa = a tratada errou menos.
- R²: o contrário; diferença positiva = a tratada foi melhor.

In [7]:
caminho_tratada = BASE_DIR / "data" / "output" / "imputacoes_outliers_tratados.xlsx"
resumo_tratada = pd.read_excel(caminho_tratada, sheet_name="resumo")
rodadas_tratada = pd.read_excel(caminho_tratada, sheet_name="rodadas")

metodos = ["media", "mediana", "knn"]
linhas = []

for metrica in ["rmse", "mae", "r2"]:
    # pivot reorganiza cada resumo: variáveis nas linhas, métodos nas colunas
    tratada = resumo_tratada.pivot(index="variavel", columns="metodo", values=metrica)
    nao_tratada = resumo.pivot(index="variavel", columns="metodo", values=metrica)

    # Variáveis na ordem da base. As colunas são preenchidas pelo índice (nome da variável)
    tabela_metrica = pd.DataFrame(index=variaveis)
    for metodo in metodos:
        tabela_metrica[f"{metodo} tratada"] = tratada[metodo]
        tabela_metrica[f"{metodo} não tratada"] = nao_tratada[metodo]
        tabela_metrica[f"{metodo} diferença"] = tratada[metodo] - nao_tratada[metodo]

        # Versão longa, para gravar no Excel: uma linha por métrica × método × variável
        for variavel in variaveis:
            linhas.append({
                "metrica": metrica,
                "metodo": metodo,
                "variavel": variavel,
                "tratada": tratada.at[variavel, metodo],
                "nao_tratada": nao_tratada.at[variavel, metodo],
                "diferenca": tratada.at[variavel, metodo] - nao_tratada.at[variavel, metodo]
            })

    # display mostra a tabela formatada, como a última linha de uma célula, mesmo dentro do laço
    print(metrica.upper())
    display(tabela_metrica.round(3))

comparacao = pd.DataFrame(linhas)

RMSE


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,85.262,85.985,-0.723,92.787,91.989,0.798,107.922,107.922,-0.000
TURB.,70.938,70.765,0.173,80.094,79.618,0.475,71.656,71.656,0.000
pH,0.153,0.153,0.001,0.158,0.166,-0.008,0.175,0.175,-0.000
ALC.,18.989,19.483,-0.494,18.799,18.790,0.010,17.920,17.929,-0.008
AC.,5.191,5.180,0.011,5.497,5.497,0.000,6.165,6.165,-0.000
O.C.,1.830,1.878,-0.047,1.831,1.832,-0.001,2.746,2.746,-0.000
O.D.,0.822,0.822,-0.001,0.821,0.821,0.000,0.904,0.904,0.000
Cl,13.829,14.358,-0.530,13.781,13.781,-0.000,13.522,13.522,-0.000
DUR.,14.833,14.856,-0.023,15.195,15.032,0.162,14.883,14.883,0.000
Fe,1.655,1.664,-0.009,1.768,1.754,0.014,1.644,1.644,0.000


MAE


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,67.399,68.990,-1.591,63.867,63.813,0.053,67.733,67.733,0.000
TURB.,56.683,57.401,-0.717,53.620,53.467,0.153,49.486,49.486,0.000
pH,0.122,0.123,-0.001,0.119,0.129,-0.009,0.138,0.138,-0.000
ALC.,16.134,16.739,-0.605,15.580,15.553,0.027,13.525,13.530,-0.005
AC.,3.643,3.813,-0.169,3.460,3.460,0.000,4.370,4.371,-0.001
O.C.,1.467,1.514,-0.046,1.465,1.471,-0.006,1.960,1.961,-0.000
O.D.,0.653,0.653,-0.000,0.649,0.649,0.000,0.716,0.716,0.000
Cl,10.441,11.532,-1.092,10.187,10.187,0.000,9.921,9.921,0.000
DUR.,12.103,12.275,-0.173,12.213,12.360,-0.147,11.362,11.360,0.002
Fe,1.330,1.354,-0.024,1.219,1.220,-0.001,1.098,1.098,0.000


R2


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,-0.020,-0.037,0.017,-0.208,-0.187,-0.021,-0.107,-0.107,0.000
TURB.,-0.016,-0.011,-0.005,-0.295,-0.280,-0.015,-0.053,-0.053,0.000
pH,-0.028,-0.021,-0.007,-0.087,-0.198,0.111,-0.267,-0.270,0.003
ALC.,-0.026,-0.080,0.054,-0.006,-0.005,-0.001,0.118,0.118,0.001
AC.,-0.019,-0.014,-0.004,-0.142,-0.142,0.000,-0.489,-0.489,0.000
O.C.,-0.030,-0.084,0.054,-0.031,-0.032,0.001,-0.424,-0.424,0.000
O.D.,-0.043,-0.045,0.001,-0.043,-0.043,0.000,-0.200,-0.200,0.000
Cl,-0.019,-0.098,0.080,-0.012,-0.012,0.000,0.005,0.005,0.000
DUR.,-0.019,-0.022,0.003,-0.069,-0.046,-0.023,-0.041,-0.041,-0.000
Fe,-0.027,-0.038,0.011,-0.172,-0.153,-0.019,0.220,0.220,0.000


# Rodadas ganhas
Com a mesma prova, cada rodada compara as duas bases nas mesmas células. Para cada método e variável: em quantas das 30 rodadas a tratada errou menos, em quantas empataram e em quantas a não tratada errou menos. Só RMSE e MAE: o R² de uma rodada oscila demais (é o motivo de juntar as rodadas).

In [8]:
# merge junta as rodadas das duas bases pela mesma seed, método e variável.
# suffixes diferencia as colunas que têm o mesmo nome nas duas tabelas (rmse_tratada, rmse_nao_tratada...)
pares = rodadas_tratada.merge(tabela, on=["seed", "metodo", "variavel"], suffixes=("_tratada", "_nao_tratada"))
print("Pares (seed × método × variável):", len(pares))

linhas = []

# Cada grupo = as 30 rodadas de um método e de uma variável
for (metodo, variavel), grupo in pares.groupby(["metodo", "variavel"], sort=False):
    for metrica in ["rmse", "mae"]:
        erro_tratada = grupo[f"{metrica}_tratada"]
        erro_nao_tratada = grupo[f"{metrica}_nao_tratada"]

        # Empate: o mesmo erro nas duas bases. np.isclose ignora diferenças minúsculas de
        # arredondamento (os números da tratada passaram pelo Excel)
        empate = np.isclose(erro_tratada, erro_nao_tratada)

        linhas.append({
            "metrica": metrica,
            "metodo": metodo,
            "variavel": variavel,
            "tratada_menor": ((erro_tratada < erro_nao_tratada) & ~empate).sum(),
            "empate": empate.sum(),
            "nao_tratada_menor": ((erro_tratada > erro_nao_tratada) & ~empate).sum()
        })

rodadas_ganhas = pd.DataFrame(linhas)

Pares (seed × método × variável): 1440


Uma tabela por métrica: variáveis nas linhas; para cada método, as três contagens (somam 30). A última linha soma as 16 variáveis (16 × 30 = 480 comparações por método).

In [9]:
for metrica in ["rmse", "mae"]:
    parte = rodadas_ganhas[rodadas_ganhas["metrica"] == metrica]

    tabela_ganhas = pd.DataFrame(index=variaveis)
    for metodo in metodos:
        # Linhas deste método, com a variável como índice, para preencher a tabela pela variável
        do_metodo = parte[parte["metodo"] == metodo].set_index("variavel")
        tabela_ganhas[f"{metodo}: tratada menor"] = do_metodo["tratada_menor"]
        tabela_ganhas[f"{metodo}: empate"] = do_metodo["empate"]
        tabela_ganhas[f"{metodo}: não tratada menor"] = do_metodo["nao_tratada_menor"]

    # Linha nova "Total": a soma de cada coluna
    tabela_ganhas.loc["Total"] = tabela_ganhas.sum()

    print(metrica.upper(), "(rodadas)")
    display(tabela_ganhas)

RMSE (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,18,0,12,5,0,25,0,30,0
TURB.,15,0,15,0,3,27,0,30,0
pH,14,0,16,4,26,0,2,28,0
ALC.,21,0,9,1,26,3,1,29,0
AC.,15,0,15,0,30,0,1,29,0
O.C.,20,0,10,15,1,14,1,29,0
O.D.,20,0,10,0,30,0,0,30,0
Cl,21,0,9,0,30,0,0,30,0
DUR.,15,0,15,12,0,18,0,29,1
Fe,19,0,11,4,7,19,0,30,0


MAE (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,22,0,8,14,0,16,0,30,0
TURB.,23,0,7,7,3,20,0,30,0
pH,19,0,11,4,26,0,2,28,0
ALC.,19,0,11,1,26,3,1,29,0
AC.,20,0,10,0,30,0,1,29,0
O.C.,18,0,12,17,0,13,1,29,0
O.D.,18,0,12,0,30,0,0,30,0
Cl,21,0,9,0,30,0,0,30,0
DUR.,16,0,14,14,0,16,0,29,1
Fe,20,0,10,11,7,12,0,30,0


Gravação dos resultados: aba `resumo` e aba `rodadas` (como no notebook da tratada), aba `comparacao` (uma linha por métrica × método × variável) e aba `rodadas_ganhas`. Como as seeds são fixas, rodar de novo dá os mesmos números.

In [10]:
caminho = BASE_DIR / "data" / "output" / "imputacoes_outliers_nao_tratados.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo
with pd.ExcelWriter(caminho) as arquivo:
    resumo.to_excel(arquivo, sheet_name="resumo", index=False)
    tabela.to_excel(arquivo, sheet_name="rodadas", index=False)
    comparacao.to_excel(arquivo, sheet_name="comparacao", index=False)
    rodadas_ganhas.to_excel(arquivo, sheet_name="rodadas_ganhas", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2\data\output\imputacoes_outliers_nao_tratados.xlsx
